# Project 15 — Revised Figure and Table Builder for Colab

This notebook is the revised, editable Colab version of the Project 15 builder.

## What this version changes

- Uses the revised Google Drive paths requested for the archived pickle and outputs.
- Saves each **composite figure** and each **panel** separately.
- Saves panel-level **PNG/PDF**, **pickle data**, **CSV** when possible, and **metadata JSON**.
- Improves spacing, panel labels, and legend placement to reduce overlaps.
- Moves the former main-text consistency figure to the **SI**.
- Builds a new main-text composite from **S3 + S5 + S8 + S10** as the new Figure 4.
- Renumbers downstream manuscript figures accordingly.

## Recommended run order

1. Mount Google Drive.
2. Run the configuration and helper cells.
3. Run all builder cells.
4. Run the final `main()` cell.
5. Inspect the output folders inside `project15_extracted`.


In [1]:
# Optional but recommended in Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
"""
Project 15 — Revised figure/table builder for Google Colab
=========================================================

This revised builder is designed for manuscript/SI redrawing with editability in mind.

What is new relative to the earlier version
-------------------------------------------
1. Uses the revised Google Drive paths requested by the user.
2. Saves each composite figure and each panel separately.
3. Saves panel-level data and panel-level metadata separately so each panel can be re-edited later.
4. Writes figure-level metadata/manifests to make downstream editing easier.
5. Improves legend placement and spacing to avoid overlaps.
6. Moves the former main-text consistency figure to the SI.
7. Creates a new main-text composite from S3 + S5 + S8 + S10:
      "Descriptor interdependence, threshold robustness, feature importance, and shortlist applicability".
8. Renumbers the downstream main-text figures accordingly:
      - Figure 4 = new descriptor/robustness/utility composite
      - Figure 5 = chemistry-facing interpretation
      - Figure 6 = application-facing evaluation

Source of truth
---------------
Only the archived bundle is used as the source of truth:
    project15_master_results_latest.pkl

Editability philosophy
----------------------
For every composite figure and for every panel, this script saves:
- PNG and PDF graphics
- panel-level pickle data
- panel-level CSV when the underlying object is tabular
- panel-level metadata JSON
- figure-level metadata JSON
- a master build manifest JSON

This makes it straightforward to:
- redraw one panel only
- edit labels or styling later
- inspect the exact data used in any panel
- update manuscript numbering without rerunning the full workflow
"""

from __future__ import annotations

import json
import os
import pickle
import textwrap
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import FancyBboxPatch

In [ ]:
# ============================================================================
# 0. USER CONFIGURATION

In [3]:
# ============================================================================
PICKLE_PATH = "/content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_master_results_latest.pkl"
OUT_DIR = Path("/content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_extracted")
DPI = 320
SAVE_PNG = True
SAVE_PDF = True

In [ ]:
# ============================================================================
# 1. OUTPUT TREE

In [4]:
# ============================================================================
DIRS = {
    "manifests": OUT_DIR / "00_manifests",
    "metadata": OUT_DIR / "01_metadata",
    "panel_data": OUT_DIR / "02_panel_data",
    "panel_metadata": OUT_DIR / "03_panel_metadata",
    "panel_png": OUT_DIR / "04_panel_png",
    "panel_pdf": OUT_DIR / "05_panel_pdf",
    "figure_data": OUT_DIR / "06_figure_data",
    "figure_metadata": OUT_DIR / "07_figure_metadata",
    "main_png": OUT_DIR / "08_main_figures_png",
    "main_pdf": OUT_DIR / "09_main_figures_pdf",
    "si_png": OUT_DIR / "10_si_figures_png",
    "si_pdf": OUT_DIR / "11_si_figures_pdf",
    "tables_csv": OUT_DIR / "12_tables_csv",
    "tables_xlsx": OUT_DIR / "13_tables_xlsx",
    "tables_tex": OUT_DIR / "14_tables_tex",
    "tables_pkl": OUT_DIR / "15_tables_pkl",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================================
# 2. STYLE

In [5]:
# ============================================================================
PALETTE = {
    "navy": "#17324D",
    "blue": "#2F6EA3",
    "sky": "#72A7CF",
    "teal": "#2B7A78",
    "green": "#4F8A5B",
    "olive": "#7A8F3A",
    "gold": "#C79A32",
    "amber": "#D9842B",
    "brick": "#A64942",
    "plum": "#7A4B7E",
    "grey": "#6E7781",
    "slate": "#75808C",
    "lightgrey": "#D7DCE2",
    "black": "#1C1C1C",
    "white": "#FFFFFF",
}
REGIME_ORDER = [
    "A_metal_only",
    "B_metal_oms",
    "C_metal_oms_context",
    "D_context_thermophysical",
]
REGIME_LABELS = {
    "A_metal_only": "Regime A",
    "B_metal_oms": "Regime B",
    "C_metal_oms_context": "Regime C",
    "D_context_thermophysical": "Regime D",
}
REGIME_COLORS = {
    "A_metal_only": PALETTE["slate"],
    "B_metal_oms": PALETTE["plum"],
    "C_metal_oms_context": PALETTE["teal"],
    "D_context_thermophysical": PALETTE["gold"],
}
TARGET_LABELS = {
    "Solvent_stability": "Solvent stability",
    "Water_stability": "Water stability",
    "Thermal_stability (℃)": "Thermal stability (°C)",
}
DATASET_COLORS = {"ASR": PALETTE["blue"], "FSR": PALETTE["brick"], "ION": PALETTE["plum"]}

sns.set_theme(style="whitegrid", context="talk")
mpl.rcParams.update({
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelweight": "medium",
    "axes.facecolor": "#FBFCFD",
    "figure.facecolor": "white",
    "grid.color": "#D6DBE2",
    "grid.linewidth": 0.8,
    "font.family": "DejaVu Sans",
    "legend.frameon": True,
    "legend.facecolor": "white",
    "legend.edgecolor": "#CCD3DB",
})

In [ ]:
# ============================================================================
# 3. HELPERS

In [6]:
# ============================================================================
def sanitize_name(text: str) -> str:
    out = (
        str(text)
        .replace(" ", "_")
        .replace("/", "-")
        .replace("(", "")
        .replace(")", "")
        .replace("℃", "C")
        .replace("°", "deg")
        .replace("Å", "A")
        .replace(":", "")
        .replace("|", "_")
        .replace(",", "")
    )
    while "__" in out:
        out = out.replace("__", "_")
    return out


def ensure_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def target_label(x: str) -> str:
    return TARGET_LABELS.get(x, x)


def regime_label(x: str) -> str:
    return REGIME_LABELS.get(x, x)


def load_pickle(path: str | Path) -> Any:
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Pickle not found: {path}")
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pickle(obj: Any, path: Path) -> None:
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def dataframe_to_csv_if_possible(obj: Any, path: Path) -> Optional[str]:
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
        return str(path)
    if isinstance(obj, pd.Series):
        obj.to_frame(name=obj.name if obj.name else "value").to_csv(path, index=True)
        return str(path)
    return None


def panel_tag(ax: plt.Axes, label: str, x: float = -0.18, y: float = 1.10) -> None:
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=16, fontweight="bold",
        color=PALETTE["navy"],
        clip_on=False,
    )


def outside_legend(ax: plt.Axes, where: str = "right", ncol: int = 1, fontsize: int = 10):
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return None
    if where == "right":
        leg = ax.legend(handles, labels, bbox_to_anchor=(1.02, 1.0), loc="upper left", borderaxespad=0, fontsize=fontsize, ncol=ncol)
    elif where == "below":
        leg = ax.legend(handles, labels, bbox_to_anchor=(0.5, -0.28), loc="upper center", borderaxespad=0, fontsize=fontsize, ncol=ncol)
    else:
        leg = ax.legend(fontsize=fontsize, ncol=ncol)
    return leg


def wrap_text(text: str, width: int = 45) -> str:
    return "\n".join(textwrap.wrap(str(text), width=width))


def ranked_barh(ax: plt.Axes, df: pd.DataFrame, y: str, x: str, title: str, color: str, xlabel: Optional[str] = None):
    temp = df.sort_values(x, ascending=True)
    ax.barh(temp[y], temp[x], color=color, edgecolor="white", linewidth=0.8)
    ax.set_title(title, loc="left", fontsize=15, pad=10)
    ax.set_xlabel(xlabel if xlabel else x)
    ax.grid(axis="y", visible=False)


def save_panel_assets(
    figure_group: str,
    panel_id: str,
    panel_fig: plt.Figure,
    panel_data: Any,
    panel_meta: Dict[str, Any],
) -> Dict[str, str]:
    stem = sanitize_name(f"{figure_group}__{panel_id}")
    out = {}
    if SAVE_PNG:
        p = DIRS["panel_png"] / f"{stem}.png"
        panel_fig.savefig(p, bbox_inches="tight")
        out["panel_png"] = str(p)
    if SAVE_PDF:
        p = DIRS["panel_pdf"] / f"{stem}.pdf"
        panel_fig.savefig(p, bbox_inches="tight")
        out["panel_pdf"] = str(p)
    pkl = DIRS["panel_data"] / f"{stem}.pkl"
    save_pickle(panel_data, pkl)
    out["panel_pkl"] = str(pkl)
    csv = dataframe_to_csv_if_possible(panel_data, DIRS["panel_data"] / f"{stem}.csv")
    if csv:
        out["panel_csv"] = csv
    meta_path = DIRS["panel_metadata"] / f"{stem}.json"
    save_json(panel_meta, meta_path)
    out["panel_meta"] = str(meta_path)
    return out


def save_composite_assets(
    figure_name: str,
    kind: str,
    fig: plt.Figure,
    composite_data: Any,
    figure_meta: Dict[str, Any],
) -> Dict[str, str]:
    stem = sanitize_name(figure_name)
    out = {}
    if kind == "main":
        png_dir, pdf_dir = DIRS["main_png"], DIRS["main_pdf"]
    else:
        png_dir, pdf_dir = DIRS["si_png"], DIRS["si_pdf"]
    if SAVE_PNG:
        p = png_dir / f"{stem}.png"
        fig.savefig(p, bbox_inches="tight")
        out["png"] = str(p)
    if SAVE_PDF:
        p = pdf_dir / f"{stem}.pdf"
        fig.savefig(p, bbox_inches="tight")
        out["pdf"] = str(p)
    data_path = DIRS["figure_data"] / f"{stem}.pkl"
    meta_path = DIRS["figure_metadata"] / f"{stem}.json"
    save_pickle(composite_data, data_path)
    save_json(figure_meta, meta_path)
    out["figure_pkl"] = str(data_path)
    out["figure_meta"] = str(meta_path)
    return out


def export_table(df: pd.DataFrame, stem: str, float_fmt: str = "%.4f") -> Dict[str, str]:
    stem = sanitize_name(stem)
    csv_path = DIRS["tables_csv"] / f"{stem}.csv"
    xlsx_path = DIRS["tables_xlsx"] / f"{stem}.xlsx"
    tex_path = DIRS["tables_tex"] / f"{stem}.tex"
    pkl_path = DIRS["tables_pkl"] / f"{stem}.pkl"
    df.to_csv(csv_path, index=False)
    df.to_excel(xlsx_path, index=False)
    df.to_pickle(pkl_path)
    df.to_latex(
        tex_path,
        index=False,
        escape=False,
        float_format=lambda x: float_fmt % x if isinstance(x, (float, np.floating)) else str(x),
    )
    return {"csv": str(csv_path), "xlsx": str(xlsx_path), "tex": str(tex_path), "pkl": str(pkl_path)}


def add_build_note(meta: Dict[str, Any], narrative: str, source_tables: Sequence[str], source_columns: Sequence[str]) -> Dict[str, Any]:
    meta = dict(meta)
    meta["narrative_role"] = narrative
    meta["source_tables"] = list(source_tables)
    meta["source_columns"] = list(source_columns)
    return meta

In [ ]:
# ============================================================================
# 4. LOAD ARCHIVE

In [7]:
# ============================================================================
bundle = load_pickle(PICKLE_PATH)
metadata = bundle["metadata"]
datasets = bundle["datasets"]
tables = bundle["tables"]
figure_support = bundle["figures_support_tables"]
si_support = bundle["si_support_tables"]

asr_clean = datasets["asr_clean"].copy()
fsr_clean = datasets["fsr_clean"].copy()
ion_clean = datasets["ion_clean"].copy()
matched_pairs = datasets["matched_pairs"].copy()
audit_df = datasets["audit_df"].copy()

regression_results_df = tables["regression_results_df"].copy()
regression_predictions_df = tables["regression_predictions_df"].copy()
regression_summary = tables["regression_summary"].copy()
classification_results_df = tables["classification_results_df"].copy()
classification_predictions_df = tables["classification_predictions_df"].copy()
classification_summary = tables["classification_summary"].copy()
transfer_results_df = tables["transfer_results_df"].copy()
permutation_importance_df = tables["permutation_importance_df"].copy()
calibration_df = tables["calibration_df"].copy()
coverage_df = tables["coverage_df"].copy()
shortlist_df = tables["shortlist_df"].copy()
chemistry_subgroup_df = tables["chemistry_subgroup_df"].copy()
failure_cases_df = tables["failure_cases_df"].copy()
best_regression_rows = tables["best_regression_rows"].copy()

In [ ]:
# ============================================================================
# 5. PANEL DRAWERS

In [8]:
# ============================================================================
def draw_flow_diagram_panel(ax: plt.Axes):
    ax.axis("off")
    panel_tag(ax, "(a)", x=-0.06, y=1.06)
    ax.set_title("Dataset flow diagram", loc="left", fontsize=15, pad=10)

    boxes = [
        (0.03, 0.82, 0.24, 0.11, "ASR_data_SI\ncore benchmark table", DATASET_COLORS["ASR"]),
        (0.03, 0.66, 0.24, 0.11, "FSR_data_SI\ncompanion benchmark table", DATASET_COLORS["FSR"]),
        (0.03, 0.50, 0.24, 0.11, "ION_data_SI\naudit-only resource", DATASET_COLORS["ION"]),
        (0.03, 0.31, 0.24, 0.11, "ASR_FSR_check.csv\nmatched-pair audit", PALETTE["teal"]),
        (0.03, 0.15, 0.24, 0.11, "recommended-screening-list.csv\napplication shortlist", PALETTE["olive"]),
        (0.38, 0.58, 0.24, 0.17, "cleaning + harmonization\nmetal parsing\nOMS parsing\ntype auditing", PALETTE["navy"]),
        (0.69, 0.58, 0.22, 0.17, "descriptor ladder\nRegime A → B → C → D", PALETTE["gold"]),
        (0.66, 0.18, 0.25, 0.20, "regression\nclassification\ntransfer audit\nsubgroup & failure analysis", PALETTE["brick"]),
    ]
    for x, y, w, h, txt, color in boxes:
        patch = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.012,rounding_size=0.02",
            linewidth=1.4, edgecolor=color,
            facecolor=mpl.colors.to_rgba(color, 0.10),
            transform=ax.transAxes,
        )
        ax.add_patch(patch)
        ax.text(x + w/2, y + h/2, txt, transform=ax.transAxes, ha="center", va="center", fontsize=11.5)
    arrows = [
        ((0.27, 0.875), (0.38, 0.665)),
        ((0.27, 0.715), (0.38, 0.665)),
        ((0.27, 0.555), (0.38, 0.665)),
        ((0.27, 0.365), (0.66, 0.28)),
        ((0.27, 0.205), (0.66, 0.28)),
        ((0.62, 0.665), (0.69, 0.665)),
        ((0.80, 0.58), (0.78, 0.40)),
    ]
    for (x1, y1), (x2, y2) in arrows:
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), xycoords=ax.transAxes, textcoords=ax.transAxes,
                    arrowprops=dict(arrowstyle="-|>", lw=1.6, color=PALETTE["slate"]))
    return {
        "counts": {
            "ASR": int(len(asr_clean)),
            "FSR": int(len(fsr_clean)),
            "ION": int(len(ion_clean)),
            "Matched_pairs": int(len(matched_pairs)),
            "Recommended_screening_list": int(shortlist_df["n"].max()) if not shortlist_df.empty else None,
        },
        "descriptor_regimes": REGIME_LABELS,
    }


def draw_target_distribution_panel(ax: plt.Axes):
    panel_tag(ax, "(b)", x=-0.12, y=1.06)
    ax.set_title("Target distributions", loc="left", fontsize=15, pad=10)
    specs = [
        (asr_clean, "Solvent_stability", "ASR: Solvent", DATASET_COLORS["ASR"]),
        (asr_clean, "Water_stability", "ASR: Water", PALETTE["amber"]),
        (asr_clean, "Thermal_stability (℃)", "ASR: Thermal (°C)", PALETTE["green"]),
        (fsr_clean, "Solvent_stability", "FSR: Solvent", DATASET_COLORS["FSR"]),
        (fsr_clean, "Water_stability", "FSR: Water", PALETTE["plum"]),
        (fsr_clean, "Thermal_stability (℃)", "FSR: Thermal (°C)", PALETTE["slate"]),
    ]
    rows = []
    for df, col, label, color in specs:
        if col in df.columns:
            vals = ensure_numeric(df[col]).dropna()
            if len(vals):
                sns.histplot(vals, bins=22, stat="count", element="step", fill=False, linewidth=2.0, ax=ax, label=label, color=color)
                rows.append({"label": label, "column": col, "n": int(len(vals))})
    ax.set_xlabel("Target value")
    ax.set_ylabel("Count")
    outside_legend(ax, where="right", fontsize=9)
    return pd.DataFrame(rows)


def draw_overlap_panel(ax: plt.Axes):
    panel_tag(ax, "(c)", x=-0.06, y=1.06)
    ax.set_title("Overlap map", loc="left", fontsize=15, pad=10)
    df = pd.DataFrame({
        "Group": ["ASR only", "FSR only", "Matched by crosswalk"],
        "Count": [len(asr_clean), len(fsr_clean), len(matched_pairs)],
    })
    sns.barplot(data=df, x="Group", y="Count", ax=ax, palette=[DATASET_COLORS["ASR"], DATASET_COLORS["FSR"], PALETTE["teal"]])
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=16)
    ymax = df["Count"].max()
    for i, row in df.iterrows():
        ax.text(i, row["Count"] + ymax*0.02, f"{int(row['Count'])}", ha="center", va="bottom", fontsize=10.5)
    return df


def draw_descriptor_ladder_panel(ax: plt.Axes):
    ax.axis("off")
    panel_tag(ax, "(d)", x=-0.12, y=1.06)
    ax.set_title("Descriptor ladder", loc="left", fontsize=15, pad=10)
    items = [
        ("Regime A: metal identity only", "primary_metal, is_mixed_metal, n_metals", REGIME_COLORS["A_metal_only"]),
        ("Regime B: A + OMS", "Has OMS, OMS Types", REGIME_COLORS["B_metal_oms"]),
        ("Regime C: B + minimal structural context", "dimension, topology, catenation, density, LCD, PLD, VF, PV", REGIME_COLORS["C_metal_oms_context"]),
        ("Regime D: C + thermophysical terms", "average_atomic_mass, heat capacities, cp parameters, natoms", REGIME_COLORS["D_context_thermophysical"]),
    ]
    y0 = 0.86
    for title, detail, color in items:
        patch = FancyBboxPatch((0.03, y0-0.12), 0.94, 0.15,
                               boxstyle="round,pad=0.015,rounding_size=0.02",
                               linewidth=1.4, edgecolor=color,
                               facecolor=mpl.colors.to_rgba(color, 0.08),
                               transform=ax.transAxes)
        ax.add_patch(patch)
        ax.text(0.06, y0-0.015, wrap_text(title, 36), transform=ax.transAxes, ha="left", va="center", fontsize=12.5, fontweight="bold")
        ax.text(0.06, y0-0.085, wrap_text(detail, 58), transform=ax.transAxes, ha="left", va="center", fontsize=10.5)
        y0 -= 0.22
    return pd.DataFrame(items, columns=["title", "details", "color"])


def draw_descriptor_ladder_main_panel(ax: plt.Axes):
    panel_tag(ax, "(a)", x=-0.10, y=1.05)
    ax.set_title("Regression descriptor ladder", loc="left", fontsize=15, pad=10)
    reg_best = (
        regression_summary.sort_values(["dataset", "target", "regime", "Spearman"], ascending=[True, True, True, False])
        .groupby(["dataset", "target", "regime"], as_index=False)
        .head(1)
        .copy()
    )
    reg_best["regime"] = pd.Categorical(reg_best["regime"], categories=REGIME_ORDER, ordered=True)
    reg_best = reg_best.sort_values(["dataset", "target", "regime"])
    for (dataset, target), sub in reg_best.groupby(["dataset", "target"]):
        ax.plot(sub["regime"].astype(str), sub["Spearman"], marker="o", linewidth=2.2, markersize=7, label=f"{dataset}: {target_label(target)}")
    ax.set_xlabel("")
    ax.set_ylabel("Spearman")
    ax.set_xticks(range(len(REGIME_ORDER)), [regime_label(r) for r in REGIME_ORDER], rotation=18)
    outside_legend(ax, where="above" if False else "right", fontsize=8)
    return reg_best


def draw_classification_heatmap_panel(ax: plt.Axes):
    panel_tag(ax, "(b)", x=-0.10, y=1.05)
    ax.set_title("ASR water-stability screening heatmap", loc="left", fontsize=15, pad=10)
    cls_best = (
        classification_summary.sort_values(["dataset", "target", "screening_cutoff", "regime", "ROC_AUC"], ascending=[True, True, True, True, False])
        .groupby(["dataset", "target", "screening_cutoff", "regime"], as_index=False)
        .head(1)
        .copy()
    )
    temp = cls_best[(cls_best["dataset"] == "ASR") & (cls_best["target"] == "Water_stability")].copy()
    heat = temp.pivot(index="regime", columns="screening_cutoff", values="ROC_AUC").reindex(REGIME_ORDER)
    sns.heatmap(heat, annot=True, fmt=".3f", cmap="mako", ax=ax, cbar_kws={"label": "ROC-AUC"})
    ax.set_ylabel("Regime")
    ax.set_xlabel("Cutoff")
    ax.set_yticklabels([regime_label(t.get_text()) for t in ax.get_yticklabels()], rotation=0)
    return temp


def draw_grouped_penalty_panel(ax: plt.Axes):
    panel_tag(ax, "(c)", x=-0.10, y=1.05)
    ax.set_title("Performance penalties under grouped evaluation", loc="left", fontsize=15, pad=10)
    rr = regression_results_df.copy()
    rr["split_family"] = np.select(
        [
            rr["split_name"].str.contains("random", na=False),
            rr["split_name"].str.contains("group_primary_metal", na=False),
            rr["split_name"].str.contains("group_topology", na=False),
            rr["split_name"].str.contains("group_Year", na=False),
        ],
        ["random", "grouped by metal", "grouped by topology", "grouped by year"],
        default="other",
    )
    best_combo = (
        regression_summary.sort_values(["dataset", "target", "Spearman"], ascending=[True, True, False])
        .groupby(["dataset", "target"], as_index=False)
        .head(1)[["dataset", "target", "regime", "model"]]
    )
    rr_best = rr.merge(best_combo, on=["dataset", "target", "regime", "model"], how="inner")
    penalty_df = rr_best.groupby(["dataset", "target", "split_family"], as_index=False)["Spearman"].mean()
    penalty_df = penalty_df[penalty_df["dataset"] == "ASR"].copy()
    sns.barplot(data=penalty_df, x="target", y="Spearman", hue="split_family", ax=ax,
                palette=[PALETTE["blue"], PALETTE["brick"], PALETTE["teal"], PALETTE["slate"]])
    ax.set_xlabel("")
    ax.set_ylabel("Mean Spearman")
    ax.set_xticklabels([target_label(t.get_text()) for t in ax.get_xticklabels()], rotation=18)
    outside_legend(ax, where="right", fontsize=8)
    return penalty_df


def draw_transfer_panel(ax: plt.Axes):
    panel_tag(ax, "(d)", x=-0.10, y=1.05)
    ax.set_title("ASR↔FSR transfer on water stability", loc="left", fontsize=15, pad=10)
    tr_best = (
        transfer_results_df.sort_values(["direction", "target", "regime", "spearman_rho"], ascending=[True, True, True, False])
        .groupby(["direction", "target", "regime"], as_index=False)
        .head(1)
        .copy()
    )
    tr_best = tr_best[tr_best["target"] == "Water_stability"].copy()
    tr_best["regime"] = pd.Categorical(tr_best["regime"], categories=REGIME_ORDER, ordered=True)
    tr_best = tr_best.sort_values(["direction", "regime"])
    for direction, sub in tr_best.groupby("direction"):
        ax.plot(sub["regime"].astype(str), sub["spearman_rho"], marker="o", linewidth=2.2, markersize=7, label=direction.replace("_to_", " → "))
    ax.set_xlabel("")
    ax.set_ylabel("Spearman (transfer)")
    ax.set_xticks(range(len(REGIME_ORDER)), [regime_label(r) for r in REGIME_ORDER], rotation=18)
    outside_legend(ax, where="right", fontsize=9)
    return tr_best


def draw_correlation_heatmap_panel(ax: plt.Axes, panel_letter: str = "(a)"):
    panel_tag(ax, panel_letter, x=-0.12, y=1.05)
    ax.set_title("Correlation matrix among pore/context descriptors", loc="left", fontsize=15, pad=10)
    cols = ["Density (g/cm3)", "LCD (Å)", "PLD (Å)", "VF", "PV (cm3/g)", "average_atomic_mass", "natoms"]
    cols = [c for c in cols if c in asr_clean.columns]
    corr = asr_clean[cols].apply(pd.to_numeric, errors="coerce").corr(method="spearman")
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="viridis", ax=ax, cbar_kws={"label": "Spearman correlation"})
    return corr


def draw_threshold_robustness_panel(ax: plt.Axes, panel_letter: str = "(b)"):
    panel_tag(ax, panel_letter, x=-0.12, y=1.05)
    ax.set_title("Threshold robustness heatmap", loc="left", fontsize=15, pad=10)
    temp = (
        classification_summary[(classification_summary["dataset"] == "ASR") & (classification_summary["model"] == "RandomForest")]
        .groupby(["target", "regime", "screening_cutoff"], as_index=False)["ROC_AUC"].max()
    )
    temp = temp[temp["target"].isin(["Solvent_stability", "Water_stability"])].copy()
    temp["column"] = temp["target"] + "|" + temp["screening_cutoff"].astype(str)
    heat = temp.pivot(index="regime", columns="column", values="ROC_AUC").reindex(REGIME_ORDER)
    readable_cols = []
    for col in heat.columns:
        target, cutoff = col.split("|")
        readable_cols.append(f"{target_label(target)}\ncutoff {cutoff}")
    heat.columns = readable_cols
    sns.heatmap(heat, annot=True, fmt=".3f", cmap="viridis", ax=ax, cbar_kws={"label": "ROC-AUC"})
    ax.set_xlabel("")
    ax.set_ylabel("Regime")
    ax.set_yticklabels([regime_label(t.get_text()) for t in ax.get_yticklabels()], rotation=0)
    return temp


def draw_importance_panel(ax: plt.Axes, panel_letter: str = "(c)"):
    panel_tag(ax, panel_letter, x=-0.12, y=1.05)
    ax.set_title("Top permutation importances", loc="left", fontsize=15, pad=10)
    temp = permutation_importance_df[(permutation_importance_df["dataset"] == "ASR") & (permutation_importance_df["target"] == "Water_stability")].copy()
    if "D_context_thermophysical" in temp["regime"].astype(str).unique():
        temp = temp[temp["regime"] == "D_context_thermophysical"].copy()
    temp = temp.sort_values(["importance_mean", "importance_std"], ascending=[False, True]).head(8)
    temp = temp.sort_values("importance_mean", ascending=True)
    ax.barh(temp["feature"], temp["importance_mean"], color=PALETTE["blue"], edgecolor="white", linewidth=0.8)
    ax.set_xlabel("Mean importance")
    ax.grid(axis="y", visible=False)
    return temp


def draw_screening_subset_line_panel(ax: plt.Axes, panel_letter: str = "(d)"):
    panel_tag(ax, panel_letter, x=-0.12, y=1.05)
    ax.set_title("Recommended-screening subset analysis", loc="left", fontsize=15, pad=10)
    temp = shortlist_df[(shortlist_df["dataset"] == "ASR") & (shortlist_df["subset"] == "recommended_screening_only")].copy()
    temp = temp[temp["target"].isin(["Solvent_stability", "Water_stability"])].copy()
    lines = []
    color_map = {"Solvent_stability": DATASET_COLORS["ASR"], "Water_stability": PALETTE["amber"]}
    marker_map = {"Solvent_stability": "o", "Water_stability": "X"}
    for target, sub in temp.groupby("target"):
        sub = sub.sort_values("screening_cutoff")
        ax.plot(sub["screening_cutoff"], sub["ROC_AUC"], marker=marker_map[target], markersize=9, linewidth=2.5,
                color=color_map[target], label=target_label(target))
        # add a faint span using the corresponding full-set ROC-AUC as context
        full = shortlist_df[(shortlist_df["dataset"] == "ASR") & (shortlist_df["subset"] == "full") & (shortlist_df["target"] == target)].copy().sort_values("screening_cutoff")
        if len(full) == len(sub):
            lo = np.minimum(sub["ROC_AUC"].values, full["ROC_AUC"].values)
            hi = np.maximum(sub["ROC_AUC"].values, full["ROC_AUC"].values)
            ax.fill_between(sub["screening_cutoff"].values, lo, hi, color=color_map[target], alpha=0.15)
        lines.append(sub)
    ax.set_xlabel("Screening cutoff")
    ax.set_ylabel("ROC-AUC")
    ax.set_xticks(sorted(temp["screening_cutoff"].unique()))
    outside_legend(ax, where="left" if False else "right", fontsize=10)
    ax.set_ylim(max(0.0, temp["ROC_AUC"].min() - 0.02), min(1.0, temp["ROC_AUC"].max() + 0.04))
    return pd.concat(lines, ignore_index=True)


def draw_old_consistency_scatter_solvent(ax: plt.Axes):
    panel_tag(ax, "(a)", x=-0.12, y=1.05)
    ax.set_title("Solvent stability pair scatter", loc="left", fontsize=15, pad=10)
    temp = matched_pairs[["ASR__Solvent_stability", "FSR__Solvent_stability"]].dropna().copy()
    sns.scatterplot(data=temp, x="ASR__Solvent_stability", y="FSR__Solvent_stability", s=24, color=PALETTE["blue"], ax=ax, edgecolor=None, alpha=0.85)
    lims = [temp.min().min(), temp.max().max()]
    ax.plot(lims, lims, linestyle="--", color=PALETTE["brick"], linewidth=1.6)
    ax.set_xlabel("ASR solvent stability")
    ax.set_ylabel("FSR solvent stability")
    return temp


def draw_old_consistency_scatter_water(ax: plt.Axes):
    panel_tag(ax, "(b)", x=-0.12, y=1.05)
    ax.set_title("Water stability pair scatter", loc="left", fontsize=15, pad=10)
    temp = matched_pairs[["ASR__Water_stability", "FSR__Water_stability"]].dropna().copy()
    sns.scatterplot(data=temp, x="ASR__Water_stability", y="FSR__Water_stability", s=24, color=PALETTE["teal"], ax=ax, edgecolor=None, alpha=0.85)
    lims = [temp.min().min(), temp.max().max()]
    ax.plot(lims, lims, linestyle="--", color=PALETTE["brick"], linewidth=1.6)
    ax.set_xlabel("ASR water stability")
    ax.set_ylabel("FSR water stability")
    return temp


def draw_old_consistency_threshold(ax: plt.Axes):
    panel_tag(ax, "(c)", x=-0.12, y=1.05)
    ax.set_title("Threshold agreement heatmap", loc="left", fontsize=15, pad=10)
    rows = []
    for threshold in [0.6, 0.7, 0.8]:
        for tgt, a, f in [
            ("Solvent", "ASR__Solvent_stability", "FSR__Solvent_stability"),
            ("Water", "ASR__Water_stability", "FSR__Water_stability"),
        ]:
            temp = matched_pairs[[a, f]].dropna().copy()
            agreement = ((temp[a] >= threshold).astype(int) == (temp[f] >= threshold).astype(int)).mean()
            rows.append({"threshold": threshold, "target": tgt, "agreement": agreement})
    df = pd.DataFrame(rows)
    heat = df.pivot(index="threshold", columns="target", values="agreement")
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="viridis", ax=ax, vmin=0.90, vmax=1.00, cbar_kws={"label": "Agreement"})
    ax.set_xlabel("")
    ax.set_ylabel("Threshold")
    return df


def draw_old_consistency_bland(ax: plt.Axes):
    panel_tag(ax, "(d)", x=-0.12, y=1.05)
    ax.set_title("Bland–Altman style plot (thermal stability)", loc="left", fontsize=15, pad=10)
    temp = matched_pairs[["ASR__Thermal_stability (℃)", "FSR__Thermal_stability (℃)"]].dropna().copy()
    temp["mean"] = temp.mean(axis=1)
    temp["difference"] = temp["ASR__Thermal_stability (℃)"] - temp["FSR__Thermal_stability (℃)"]
    sns.scatterplot(data=temp, x="mean", y="difference", s=24, color=PALETTE["blue"], ax=ax, edgecolor=None, alpha=0.8)
    ax.axhline(0, linestyle="--", color=PALETTE["brick"], linewidth=1.4)
    ax.set_xlabel("Mean(ASR, FSR)")
    ax.set_ylabel("ASR - FSR")
    return temp


def draw_chemistry_primary_metal(ax: plt.Axes):
    panel_tag(ax, "(a)", x=-0.12, y=1.05)
    ax.set_title("Performance by primary metal", loc="left", fontsize=15, pad=10)
    best = regression_summary[(regression_summary["dataset"] == "ASR") & (regression_summary["target"] == "Water_stability")].sort_values(["Spearman", "R2"], ascending=[False, False]).head(1)
    reg, model = best["regime"].iloc[0], best["model"].iloc[0]
    temp = chemistry_subgroup_df[(chemistry_subgroup_df["dataset"] == "ASR") & (chemistry_subgroup_df["target"] == "Water_stability") & (chemistry_subgroup_df["regime"] == reg) & (chemistry_subgroup_df["model"] == model) & (chemistry_subgroup_df["group_col"] == "primary_metal") & (chemistry_subgroup_df["n"] >= 10)].copy()
    temp = temp.sort_values("MAE", ascending=False).head(15)
    ranked_barh(ax, temp, y="group_value", x="MAE", title="Performance by primary metal", color=PALETTE["blue"], xlabel="MAE")
    return temp


def draw_chemistry_mixed(ax: plt.Axes):
    panel_tag(ax, "(b)", x=-0.12, y=1.05)
    ax.set_title("Mixed-metal vs single-metal", loc="left", fontsize=15, pad=10)
    best = regression_summary[(regression_summary["dataset"] == "ASR") & (regression_summary["target"] == "Water_stability")].sort_values(["Spearman", "R2"], ascending=[False, False]).head(1)
    reg, model = best["regime"].iloc[0], best["model"].iloc[0]
    pred = regression_predictions_df[(regression_predictions_df["dataset"] == "ASR") & (regression_predictions_df["target"] == "Water_stability") & (regression_predictions_df["regime"] == reg) & (regression_predictions_df["model"] == model)].copy()
    pred["abs_error"] = (pred["y_true"] - pred["y_pred"]).abs()
    temp = pred.groupby("is_mixed_metal", as_index=False)["abs_error"].mean()
    temp["is_mixed_metal"] = temp["is_mixed_metal"].fillna(0).astype(int)
    sns.barplot(data=temp, x="is_mixed_metal", y="abs_error", ax=ax, palette=[PALETTE["teal"], PALETTE["brick"]])
    ax.set_xlabel("is_mixed_metal")
    ax.set_ylabel("MAE")
    return temp


def draw_chemistry_oms(ax: plt.Axes):
    panel_tag(ax, "(c)", x=-0.12, y=1.05)
    ax.set_title("OMS vs non-OMS", loc="left", fontsize=15, pad=10)
    best = regression_summary[(regression_summary["dataset"] == "ASR") & (regression_summary["target"] == "Water_stability")].sort_values(["Spearman", "R2"], ascending=[False, False]).head(1)
    reg, model = best["regime"].iloc[0], best["model"].iloc[0]
    pred = regression_predictions_df[(regression_predictions_df["dataset"] == "ASR") & (regression_predictions_df["target"] == "Water_stability") & (regression_predictions_df["regime"] == reg) & (regression_predictions_df["model"] == model)].copy()
    pred["abs_error"] = (pred["y_true"] - pred["y_pred"]).abs()
    temp = pred.copy()
    temp["Has OMS"] = temp["Has OMS"].astype(str).replace({"True": "Yes", "False": "No", "1": "Yes", "0": "No"})
    temp = temp[temp["Has OMS"].isin(["Yes", "No"])].groupby("Has OMS", as_index=False)["abs_error"].mean()
    sns.barplot(data=temp, x="Has OMS", y="abs_error", order=["No", "Yes"], ax=ax, palette=[PALETTE["brick"], PALETTE["teal"]])
    ax.set_ylabel("MAE")
    return temp


def draw_chemistry_metal_oms_heat(ax: plt.Axes):
    panel_tag(ax, "(d)", x=-0.12, y=1.05)
    ax.set_title("Median stability by metal × OMS", loc="left", fontsize=15, pad=10)
    best = regression_summary[(regression_summary["dataset"] == "ASR") & (regression_summary["target"] == "Water_stability")].sort_values(["Spearman", "R2"], ascending=[False, False]).head(1)
    reg, model = best["regime"].iloc[0], best["model"].iloc[0]
    pred = regression_predictions_df[(regression_predictions_df["dataset"] == "ASR") & (regression_predictions_df["target"] == "Water_stability") & (regression_predictions_df["regime"] == reg) & (regression_predictions_df["model"] == model)].copy()
    temp = pred.copy()
    temp["Has OMS"] = temp["Has OMS"].astype(str).replace({"True": "Yes", "False": "No", "1": "Yes", "0": "No"})
    temp = temp[temp["Has OMS"].isin(["Yes", "No"])].groupby(["primary_metal", "Has OMS"], as_index=False)["y_true"].median()
    top_metals = pred["primary_metal"].astype(str).value_counts().head(10).index
    temp = temp[temp["primary_metal"].isin(top_metals)]
    heat = temp.pivot(index="primary_metal", columns="Has OMS", values="y_true")
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="viridis", ax=ax, cbar_kws={"label": "Median stability"})
    return temp


def draw_chemistry_topology(ax: plt.Axes):
    panel_tag(ax, "(e)", x=-0.12, y=1.05)
    ax.set_title("Topology-stratified performance", loc="left", fontsize=15, pad=10)
    best = regression_summary[(regression_summary["dataset"] == "ASR") & (regression_summary["target"] == "Water_stability")].sort_values(["Spearman", "R2"], ascending=[False, False]).head(1)
    reg, model = best["regime"].iloc[0], best["model"].iloc[0]
    temp = chemistry_subgroup_df[(chemistry_subgroup_df["dataset"] == "ASR") & (chemistry_subgroup_df["target"] == "Water_stability") & (chemistry_subgroup_df["regime"] == reg) & (chemistry_subgroup_df["model"] == model) & (chemistry_subgroup_df["group_col"] == "topology(AllNodes)") & (chemistry_subgroup_df["n"] >= 8)].copy()
    temp = temp.sort_values("MAE", ascending=False).head(14)
    ranked_barh(ax, temp, y="group_value", x="MAE", title="Topology-stratified performance", color=PALETTE["olive"], xlabel="MAE")
    return temp


def draw_chemistry_failure(ax: plt.Axes):
    panel_tag(ax, "(f)", x=-0.12, y=1.05)
    ax.set_title("Failure-enrichment map", loc="left", fontsize=15, pad=10)
    temp = failure_cases_df[(failure_cases_df["dataset"] == "ASR") & (failure_cases_df["target"] == "Water_stability")].copy()
    temp = temp.groupby("primary_metal", as_index=False).size().rename(columns={"size": "count"}).sort_values("count", ascending=False).head(12)
    ranked_barh(ax, temp, y="primary_metal", x="count", title="Failure-enrichment map", color=PALETTE["brick"], xlabel="Count among worst-predicted cases")
    return temp


def draw_calibration_panel(ax: plt.Axes):
    panel_tag(ax, "(a)", x=-0.12, y=1.05)
    ax.set_title("Calibration curve", loc="left", fontsize=15, pad=10)
    best = classification_summary[(classification_summary["dataset"] == "ASR") & (classification_summary["target"] == "Water_stability")].sort_values(["ROC_AUC", "PR_AUC"], ascending=[False, False]).head(1)
    cutoff, regime, model = float(best["screening_cutoff"].iloc[0]), best["regime"].iloc[0], best["model"].iloc[0]
    temp = calibration_df[(calibration_df["dataset"] == "ASR") & (calibration_df["target"] == "Water_stability") & (calibration_df["screening_cutoff"] == cutoff) & (calibration_df["regime"] == regime) & (calibration_df["model"] == model)].copy().sort_values("mean_predicted_probability")
    ax.plot([0,1],[0,1], linestyle="--", color=PALETTE["slate"], linewidth=1.4)
    ax.plot(temp["mean_predicted_probability"], temp["fraction_positive"], marker="o", linewidth=2.2, color=PALETTE["amber"])
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed positive fraction")
    return temp


def draw_coverage_panel(ax: plt.Axes):
    panel_tag(ax, "(b)", x=-0.12, y=1.05)
    ax.set_title("Confidence vs retained coverage", loc="left", fontsize=15, pad=10)
    best = classification_summary[(classification_summary["dataset"] == "ASR") & (classification_summary["target"] == "Water_stability")].sort_values(["ROC_AUC", "PR_AUC"], ascending=[False, False]).head(1)
    cutoff, regime, model = float(best["screening_cutoff"].iloc[0]), best["regime"].iloc[0], best["model"].iloc[0]
    temp = coverage_df[(coverage_df["dataset"] == "ASR") & (coverage_df["target"] == "Water_stability") & (coverage_df["screening_cutoff"] == cutoff) & (coverage_df["regime"] == regime) & (coverage_df["model"] == model)].copy().sort_values("coverage")
    ycol = "precision_at_retained" if "precision_at_retained" in temp.columns else "observed_positive_rate"
    ax.plot(temp["coverage"], temp[ycol], marker="o", linewidth=2.2, color=PALETTE["blue"])
    ax.set_xlabel("Retained coverage")
    ax.set_ylabel("Observed positive fraction")
    return temp


def draw_enrichment_panel(ax: plt.Axes):
    panel_tag(ax, "(c)", x=-0.12, y=1.05)
    ax.set_title("Enrichment in top predicted decile", loc="left", fontsize=15, pad=10)
    best = shortlist_df[(shortlist_df["dataset"] == "ASR") & (shortlist_df["target"] == "Water_stability")].sort_values("ROC_AUC", ascending=False).head(1)
    df = pd.DataFrame({
        "Group": ["Overall", "Top decile"],
        "Positive fraction": [best["overall_positive_rate"].iloc[0], best["top_decile_positive_rate"].iloc[0]],
    })
    sns.barplot(data=df, x="Group", y="Positive fraction", ax=ax, palette=[PALETTE["slate"], PALETTE["teal"]])
    ax.set_ylabel("Positive fraction")
    return df


def draw_subset_compare_panel(ax: plt.Axes):
    panel_tag(ax, "(d)", x=-0.12, y=1.05)
    ax.set_title("Recommended-screening subset", loc="left", fontsize=15, pad=10)
    best = shortlist_df[(shortlist_df["dataset"] == "ASR") & (shortlist_df["target"] == "Water_stability")].sort_values("ROC_AUC", ascending=False).head(2)
    temp = best[["subset", "ROC_AUC"]].drop_duplicates().copy()
    sns.barplot(data=temp, x="subset", y="ROC_AUC", ax=ax, palette=[PALETTE["blue"], PALETTE["olive"]])
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=18)
    ax.set_ylabel("ROC-AUC")
    return temp


def draw_workflow_panel(ax: plt.Axes):
    ax.axis("off")
    panel_tag(ax, "(e)", x=-0.02, y=1.05)
    ax.set_title("Example shortlist workflow", loc="left", fontsize=15, pad=10)
    text = (
        "1. Predict stability probability for all MOFs\n"
        "2. Restrict to the recommended-screening subset\n"
        "3. Rank by predicted probability\n"
        "4. Keep the top-confidence decile or top-N entries\n"
        "5. Review chemistry-facing flags:\n"
        "   • metal family\n"
        "   • OMS presence\n"
        "   • mixed-metal status\n"
        "   • topology\n"
        "6. Advance to experiment or simulation follow-up"
    )
    ax.text(0.01, 0.98, text, transform=ax.transAxes, ha="left", va="top", fontsize=12.5)
    return {"workflow": text}


def draw_probability_distribution_panel(ax: plt.Axes):
    panel_tag(ax, "(f)", x=-0.12, y=1.05)
    ax.set_title("Predicted probability distribution", loc="left", fontsize=15, pad=10)
    best = classification_summary[(classification_summary["dataset"] == "ASR") & (classification_summary["target"] == "Water_stability")].sort_values(["ROC_AUC", "PR_AUC"], ascending=[False, False]).head(1)
    cutoff, regime, model = float(best["screening_cutoff"].iloc[0]), best["regime"].iloc[0], best["model"].iloc[0]
    temp = calibration_df[(calibration_df["dataset"] == "ASR") & (calibration_df["target"] == "Water_stability") & (calibration_df["screening_cutoff"] == cutoff) & (calibration_df["regime"] == regime) & (calibration_df["model"] == model)].copy()
    surrogate = np.repeat(temp["mean_predicted_probability"].round(4).values, 50)
    sns.histplot(surrogate, bins=18, color=PALETTE["blue"], ax=ax)
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Count")
    return temp

In [ ]:
# ============================================================================
# 6. FIGURE WRAPPERS

In [9]:
# ============================================================================
def render_panel(panel_drawer, figure_group: str, panel_id: str, figsize=(6.5, 5.0), metadata_extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    panel_data = panel_drawer(ax)
    meta = {"figure_group": figure_group, "panel_id": panel_id}
    if metadata_extra:
        meta.update(metadata_extra)
    assets = save_panel_assets(figure_group, panel_id, fig, panel_data, meta)
    plt.close(fig)
    return {"data": panel_data, "assets": assets, "meta": meta}


def build_figure_1_dataset_architecture() -> Dict[str, Any]:
    figure_name = "Figure_1_dataset_architecture"
    kind = "main"
    fig = plt.figure(figsize=(17.2, 11.8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.35, 1.0], hspace=0.34, wspace=0.34)
    ax_a = fig.add_subplot(gs[0, 0]); data_a = draw_flow_diagram_panel(ax_a)
    ax_b = fig.add_subplot(gs[0, 1]); data_b = draw_target_distribution_panel(ax_b)
    ax_c = fig.add_subplot(gs[1, 0]); data_c = draw_overlap_panel(ax_c)
    ax_d = fig.add_subplot(gs[1, 1]); data_d = draw_descriptor_ladder_panel(ax_d)
    fig.suptitle("Figure 1. Dataset architecture and benchmark logic", fontsize=22, fontweight="bold", y=0.985)
    fig.subplots_adjust(top=0.90)
    figure_meta = add_build_note(
        {"figure_name": figure_name, "kind": kind, "panel_order": ["a", "b", "c", "d"]},
        narrative="Architecture, target coverage, overlap, and descriptor-regime logic.",
        source_tables=["datasets.asr_clean", "datasets.fsr_clean", "datasets.ion_clean", "datasets.matched_pairs"],
        source_columns=["Solvent_stability", "Water_stability", "Thermal_stability (℃)", "primary_metal", "is_mixed_metal"],
    )
    composite_data = {"panel_a": data_a, "panel_b": data_b, "panel_c": data_c, "panel_d": data_d}
    assets = save_composite_assets(figure_name, kind, fig, composite_data, figure_meta)
    plt.close(fig)
    panels = {
        "a": render_panel(draw_flow_diagram_panel, figure_name, "panel_a_flow_diagram", figsize=(10.0, 7.0)),
        "b": render_panel(draw_target_distribution_panel, figure_name, "panel_b_target_distributions", figsize=(9.0, 6.0)),
        "c": render_panel(draw_overlap_panel, figure_name, "panel_c_overlap_map", figsize=(8.0, 6.0)),
        "d": render_panel(draw_descriptor_ladder_panel, figure_name, "panel_d_descriptor_ladder", figsize=(10.0, 7.5)),
    }
    return {"assets": assets, "panels": panels, "meta": figure_meta}


def build_figure_3_descriptor_sufficiency() -> Dict[str, Any]:
    figure_name = "Figure_3_descriptor_sufficiency"
    kind = "main"
    fig, axs = plt.subplots(2, 2, figsize=(16.5, 12.5), constrained_layout=True)
    data_a = draw_descriptor_ladder_main_panel(axs[0, 0])
    data_b = draw_classification_heatmap_panel(axs[0, 1])
    data_c = draw_grouped_penalty_panel(axs[1, 0])
    data_d = draw_transfer_panel(axs[1, 1])
    fig.suptitle("Figure 3. Descriptor sufficiency across the core tasks", fontsize=21, fontweight="bold", y=1.01)
    figure_meta = add_build_note(
        {"figure_name": figure_name, "kind": kind, "panel_order": ["a", "b", "c", "d"]},
        narrative="How performance evolves across descriptor regimes, grouped evaluation, and cross-database transfer.",
        source_tables=["tables.regression_summary", "tables.classification_summary", "tables.regression_results_df", "tables.transfer_results_df"],
        source_columns=["Spearman", "ROC_AUC", "split_name", "regime", "target"],
    )
    composite_data = {"panel_a": data_a, "panel_b": data_b, "panel_c": data_c, "panel_d": data_d}
    assets = save_composite_assets(figure_name, kind, fig, composite_data, figure_meta)
    plt.close(fig)
    panels = {
        "a": render_panel(draw_descriptor_ladder_main_panel, figure_name, "panel_a_regression_ladder", figsize=(8.5, 6.5)),
        "b": render_panel(draw_classification_heatmap_panel, figure_name, "panel_b_water_heatmap", figsize=(8.5, 6.5)),
        "c": render_panel(draw_grouped_penalty_panel, figure_name, "panel_c_grouped_penalty", figsize=(9.0, 6.5)),
        "d": render_panel(draw_transfer_panel, figure_name, "panel_d_transfer", figsize=(8.5, 6.5)),
    }
    return {"assets": assets, "panels": panels, "meta": figure_meta}


def build_figure_4_descriptor_structure_robustness_utility() -> Dict[str, Any]:
    figure_name = "Figure_4_descriptor_interdependence_robustness_utility"
    kind = "main"
    fig, axs = plt.subplots(2, 2, figsize=(16.0, 12.5), constrained_layout=True)
    data_a = draw_correlation_heatmap_panel(axs[0, 0], panel_letter="(a)")
    data_b = draw_threshold_robustness_panel(axs[0, 1], panel_letter="(b)")
    data_c = draw_importance_panel(axs[1, 0], panel_letter="(c)")
    data_d = draw_screening_subset_line_panel(axs[1, 1], panel_letter="(d)")
    fig.suptitle("Figure 4. Descriptor interdependence, threshold robustness, feature importance, and shortlist applicability", fontsize=20, fontweight="bold", y=1.01)
    figure_meta = add_build_note(
        {"figure_name": figure_name, "kind": kind, "panel_order": ["a", "b", "c", "d"]},
        narrative="Bridge figure between descriptor sufficiency and downstream applicability: redundancy/complementarity, robustness, interpretability, and shortlist utility.",
        source_tables=["si_support_tables.S3_correlation_matrix", "si_support_tables.S5_threshold_robustness", "si_support_tables.S8_importance", "si_support_tables.S10_screening_subset"],
        source_columns=["Density (g/cm3)", "LCD (Å)", "PLD (Å)", "VF", "PV (cm3/g)", "average_atomic_mass", "natoms", "ROC_AUC", "importance_mean", "screening_cutoff"],
    )
    composite_data = {"panel_a": data_a, "panel_b": data_b, "panel_c": data_c, "panel_d": data_d}
    assets = save_composite_assets(figure_name, kind, fig, composite_data, figure_meta)
    plt.close(fig)
    panels = {
        "a": render_panel(lambda ax: draw_correlation_heatmap_panel(ax, panel_letter="(a)"), figure_name, "panel_a_correlation_matrix", figsize=(9.5, 7.5)),
        "b": render_panel(lambda ax: draw_threshold_robustness_panel(ax, panel_letter="(b)"), figure_name, "panel_b_threshold_robustness", figsize=(10.0, 6.5)),
        "c": render_panel(lambda ax: draw_importance_panel(ax, panel_letter="(c)"), figure_name, "panel_c_importance", figsize=(9.0, 6.0)),
        "d": render_panel(lambda ax: draw_screening_subset_line_panel(ax, panel_letter="(d)"), figure_name, "panel_d_shortlist_utility", figsize=(9.5, 6.0)),
    }
    return {"assets": assets, "panels": panels, "meta": figure_meta}


def build_figure_5_chemistry_facing() -> Dict[str, Any]:
    figure_name = "Figure_5_chemistry_regime_map"
    kind = "main"
    fig, axs = plt.subplots(3, 2, figsize=(16.5, 15.5), constrained_layout=True)
    data_a = draw_chemistry_primary_metal(axs[0, 0])
    data_b = draw_chemistry_mixed(axs[0, 1])
    data_c = draw_chemistry_oms(axs[1, 0])
    data_d = draw_chemistry_metal_oms_heat(axs[1, 1])
    data_e = draw_chemistry_topology(axs[2, 0])
    data_f = draw_chemistry_failure(axs[2, 1])
    fig.suptitle("Figure 5. Chemistry-facing interpretation of benchmark behavior", fontsize=21, fontweight="bold", y=1.01)
    figure_meta = add_build_note(
        {"figure_name": figure_name, "kind": kind, "panel_order": ["a", "b", "c", "d", "e", "f"]},
        narrative="Chemistry-facing subgroup and failure interpretation for the strongest ASR water-stability regression setting.",
        source_tables=["tables.chemistry_subgroup_df", "tables.failure_cases_df", "tables.regression_predictions_df", "tables.regression_summary"],
        source_columns=["primary_metal", "is_mixed_metal", "Has OMS", "MAE", "y_true", "y_pred", "topology(AllNodes)"],
    )
    composite_data = {"panel_a": data_a, "panel_b": data_b, "panel_c": data_c, "panel_d": data_d, "panel_e": data_e, "panel_f": data_f}
    assets = save_composite_assets(figure_name, kind, fig, composite_data, figure_meta)
    plt.close(fig)
    panels = {
        "a": render_panel(draw_chemistry_primary_metal, figure_name, "panel_a_primary_metal", figsize=(8.0, 6.2)),
        "b": render_panel(draw_chemistry_mixed, figure_name, "panel_b_mixed_metal", figsize=(7.5, 5.8)),
        "c": render_panel(draw_chemistry_oms, figure_name, "panel_c_oms", figsize=(7.0, 5.8)),
        "d": render_panel(draw_chemistry_metal_oms_heat, figure_name, "panel_d_metal_x_oms", figsize=(8.0, 6.0)),
        "e": render_panel(draw_chemistry_topology, figure_name, "panel_e_topology", figsize=(8.0, 6.2)),
        "f": render_panel(draw_chemistry_failure, figure_name, "panel_f_failure_map", figsize=(7.8, 5.8)),
    }
    return {"assets": assets, "panels": panels, "meta": figure_meta}


def build_figure_6_application_facing() -> Dict[str, Any]:
    figure_name = "Figure_6_practical_screening"
    kind = "main"
    fig, axs = plt.subplots(3, 2, figsize=(16.0, 15.0), constrained_layout=True)
    data_a = draw_calibration_panel(axs[0, 0])
    data_b = draw_coverage_panel(axs[0, 1])
    data_c = draw_enrichment_panel(axs[1, 0])
    data_d = draw_subset_compare_panel(axs[1, 1])
    data_e = draw_workflow_panel(axs[2, 0])
    data_f = draw_probability_distribution_panel(axs[2, 1])
    fig.suptitle("Figure 6. Application-facing evaluation of the classification layer", fontsize=21, fontweight="bold", y=1.01)
    figure_meta = add_build_note(
        {"figure_name": figure_name, "kind": kind, "panel_order": ["a", "b", "c", "d", "e", "f"]},
        narrative="Calibration, retained-coverage behaviour, enrichment, subset comparison, workflow, and probability distribution for the strongest ASR water-stability classification setting.",
        source_tables=["tables.calibration_df", "tables.coverage_df", "tables.shortlist_df", "tables.classification_summary"],
        source_columns=["mean_predicted_probability", "fraction_positive", "coverage", "ROC_AUC", "overall_positive_rate", "top_decile_positive_rate"],
    )
    composite_data = {"panel_a": data_a, "panel_b": data_b, "panel_c": data_c, "panel_d": data_d, "panel_e": data_e, "panel_f": data_f}
    assets = save_composite_assets(figure_name, kind, fig, composite_data, figure_meta)
    plt.close(fig)
    panels = {
        "a": render_panel(draw_calibration_panel, figure_name, "panel_a_calibration", figsize=(7.2, 5.8)),
        "b": render_panel(draw_coverage_panel, figure_name, "panel_b_coverage", figsize=(7.2, 5.8)),
        "c": render_panel(draw_enrichment_panel, figure_name, "panel_c_enrichment", figsize=(6.8, 5.6)),
        "d": render_panel(draw_subset_compare_panel, figure_name, "panel_d_subset_compare", figsize=(7.2, 5.8)),
        "e": render_panel(draw_workflow_panel, figure_name, "panel_e_workflow", figsize=(7.2, 6.0)),
        "f": render_panel(draw_probability_distribution_panel, figure_name, "panel_f_probability_distribution", figsize=(7.2, 5.8)),
    }
    return {"assets": assets, "panels": panels, "meta": figure_meta}

# --- SI wrappers ---

def build_si_s1_missingness() -> Dict[str, Any]:
    outputs = {}
    for tag, df in [("ASR", asr_clean), ("FSR", fsr_clean), ("ION", ion_clean)]:
        figure_name = f"SI_Figure_S1_missingness_{tag}"
        def drawer(ax, _df=df, _tag=tag):
            panel_tag(ax, "(S1)", x=-0.10, y=1.05)
            ax.set_title(f"S1/SI data missingness — {_tag}", loc="left", fontsize=16, pad=10)
            temp = pd.DataFrame({"column": _df.columns, "missing_fraction": _df.isna().mean().values}).sort_values("missing_fraction", ascending=False)
            sns.barplot(data=temp, x="column", y="missing_fraction", color=PALETTE["blue"], ax=ax)
            ax.set_xlabel("")
            ax.set_ylabel("Missing fraction")
            ax.tick_params(axis="x", rotation=90, labelsize=8.5)
            return temp
        outputs[tag] = render_panel(drawer, figure_name, f"panel_missingness_{tag}", figsize=(22, 5.5))
    return outputs


def build_si_s2_descriptor_distributions() -> Dict[str, Any]:
    figure_name = "SI_Figure_S2_descriptor_distributions"
    cols = ["Density (g/cm3)", "LCD (Å)", "PLD (Å)", "VF", "PV (cm3/g)", "average_atomic_mass", "natoms"]
    cols = [c for c in cols if c in asr_clean.columns]
    fig, axs = plt.subplots(len(cols), 1, figsize=(10.5, 3.2*len(cols)), constrained_layout=True)
    if len(cols) == 1:
        axs = [axs]
    composite = {}
    for i, (ax, col) in enumerate(zip(axs, cols), start=1):
        vals = ensure_numeric(asr_clean[col]).dropna()
        sns.histplot(vals, bins=25, color=PALETTE["blue"], ax=ax)
        ax.set_title(f"S2 distribution — {col}", loc="left", fontsize=14, pad=8)
        composite[col] = vals
    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Marginal distributions of key pore/context descriptors in ASR.", ["datasets.asr_clean"], cols)
    assets = save_composite_assets(figure_name, "si", fig, composite, fig_meta)
    plt.close(fig)
    panel_outputs = {}
    for col in cols:
        def drawer(ax, _col=col):
            vals = ensure_numeric(asr_clean[_col]).dropna()
            sns.histplot(vals, bins=25, color=PALETTE["blue"], ax=ax)
            ax.set_title(f"S2 distribution — {_col}", loc="left", fontsize=14, pad=8)
            ax.set_xlabel(_col)
            return pd.DataFrame({"value": vals})
        panel_outputs[sanitize_name(col)] = render_panel(drawer, figure_name, f"panel_{sanitize_name(col)}", figsize=(8.5, 4.8))
    return {"assets": assets, "panels": panel_outputs, "meta": fig_meta}


def build_si_s3_correlation_matrix() -> Dict[str, Any]:
    figure_name = "SI_Figure_S3_correlation_matrix"
    fig, ax = plt.subplots(figsize=(11.5, 8.5), constrained_layout=True)
    data = draw_correlation_heatmap_panel(ax, panel_letter="(S3)")
    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Spearman correlation matrix among selected pore/context descriptors.", ["datasets.asr_clean"], list(data.columns))
    assets = save_composite_assets(figure_name, "si", fig, {"correlation_matrix": data}, fig_meta)
    plt.close(fig)
    panel = render_panel(lambda ax: draw_correlation_heatmap_panel(ax, panel_letter="(S3)"), figure_name, "panel_s3_correlation_matrix", figsize=(11.0, 8.0))
    return {"assets": assets, "panels": {"main": panel}, "meta": fig_meta}


def build_si_s4_metal_breakdown() -> Dict[str, Any]:
    figure_name = "SI_Figure_S4_metal_breakdown"
    fig, axs = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    def panel_a(ax):
        panel_tag(ax, "(a)", x=-0.12, y=1.05)
        temp = asr_clean["primary_metal"].astype(str).value_counts().reset_index()
        temp.columns = ["primary_metal", "count"]
        temp = temp.head(14).sort_values("count", ascending=True)
        ranked_barh(ax, temp, y="primary_metal", x="count", title="S4 Primary metal frequency (ASR)", color=PALETTE["blue"], xlabel="Count")
        return temp
    def panel_b(ax):
        panel_tag(ax, "(b)", x=-0.12, y=1.05)
        temp = asr_clean["is_mixed_metal"].fillna(0).astype(int).value_counts().sort_index().reset_index()
        temp.columns = ["is_mixed_metal", "count"]
        sns.barplot(data=temp, x="is_mixed_metal", y="count", palette=[PALETTE["teal"], PALETTE["brick"]], ax=ax)
        ax.set_title("S4 Mixed-metal breakdown (ASR)", loc="left", fontsize=15, pad=10)
        return temp
    data_a = panel_a(axs[0]); data_b = panel_b(axs[1])
    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Basic metal-frequency and mixed-metal composition diagnostics for ASR.", ["datasets.asr_clean"], ["primary_metal", "is_mixed_metal"])
    assets = save_composite_assets(figure_name, "si", fig, {"panel_a": data_a, "panel_b": data_b}, fig_meta)
    plt.close(fig)
    panels = {"a": render_panel(panel_a, figure_name, "panel_a_primary_metal_frequency", figsize=(8.0, 6.0)), "b": render_panel(panel_b, figure_name, "panel_b_mixed_metal_breakdown", figsize=(7.2, 5.8))}
    return {"assets": assets, "panels": panels, "meta": fig_meta}


def build_si_s5_threshold_robustness() -> Dict[str, Any]:
    figure_name = "SI_Figure_S5_threshold_robustness"
    fig, ax = plt.subplots(figsize=(13, 6.8), constrained_layout=True)
    data = draw_threshold_robustness_panel(ax, panel_letter="(S5)")
    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Robustness of classification conclusions across reasonable screening cutoffs.", ["tables.classification_summary"], ["dataset", "target", "screening_cutoff", "regime", "model", "ROC_AUC"])
    assets = save_composite_assets(figure_name, "si", fig, {"heatmap_source": data}, fig_meta)
    plt.close(fig)
    panel = render_panel(lambda ax: draw_threshold_robustness_panel(ax, panel_letter="(S5)"), figure_name, "panel_s5_threshold_robustness", figsize=(11.0, 6.5))
    return {"assets": assets, "panels": {"main": panel}, "meta": fig_meta}


def build_si_s6_grouped_cv_details() -> Dict[str, Any]:
    figure_name = "SI_Figure_S6_grouped_cv_details"
    fig, ax = plt.subplots(figsize=(12.0, 7.0), constrained_layout=True)
    panel_tag(ax, "(S6)", x=-0.08, y=1.05)
    ax.set_title("S6 Grouped-by-primary-metal CV details", loc="left", fontsize=16, pad=10)
    temp = (
        regression_results_df[regression_results_df["split_name"].str.contains("group_primary_metal", na=False)]
        .groupby(["regime", "model"], as_index=False)["Spearman"].mean()
    )
    model_order = [m for m in ["HistGB", "RandomForest", "Ridge"] if m in temp["model"].unique()]
    temp["model"] = pd.Categorical(temp["model"], categories=model_order, ordered=True)
    temp["regime"] = pd.Categorical(temp["regime"], categories=REGIME_ORDER, ordered=True)
    temp = temp.sort_values(["regime", "model"])
    markers = {"A_metal_only": "o", "B_metal_oms": "X", "C_metal_oms_context": "s", "D_context_thermophysical": "P"}
    for regime, sub in temp.groupby("regime"):
        ax.plot(sub["model"].astype(str), sub["Spearman"], marker=markers.get(regime, "o"), linewidth=2.5, markersize=10, label=regime)
    ax.set_xlabel("")
    ax.set_ylabel("Mean Spearman")
    leg = outside_legend(ax, where="right", fontsize=10)
    if leg:
        for txt in leg.get_texts():
            txt.set_text(regime_label(txt.get_text()))
    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Grouped-by-primary-metal CV detail plot by regime and model.", ["tables.regression_results_df"], ["split_name", "regime", "model", "Spearman"])
    assets = save_composite_assets(figure_name, "si", fig, {"data": temp}, fig_meta)
    plt.close(fig)
    panel = render_panel(lambda ax: (
        panel_tag(ax, "(S6)", x=-0.08, y=1.05),
        ax.set_title("S6 Grouped-by-primary-metal CV details", loc="left", fontsize=16, pad=10),
        [ax.plot(sub["model"].astype(str), sub["Spearman"], marker=markers.get(regime, "o"), linewidth=2.5, markersize=10, label=regime) for regime, sub in temp.groupby("regime")],
        ax.set_xlabel(""), ax.set_ylabel("Mean Spearman"), outside_legend(ax, where="right", fontsize=10), temp
    )[-1], figure_name, "panel_s6_grouped_cv", figsize=(10.5, 6.5))
    return {"assets": assets, "panels": {"main": panel}, "meta": fig_meta}


def build_si_s7_old_main_figure2_consistency() -> Dict[str, Any]:
    figure_name = "SI_Figure_S7_matched_pair_diagnostics"
    fig, axs = plt.subplots(2, 2, figsize=(15.5, 12.0), constrained_layout=True)
    data_a = draw_old_consistency_scatter_solvent(axs[0, 0])
    data_b = draw_old_consistency_scatter_water(axs[0, 1])
    data_c = draw_old_consistency_threshold(axs[1, 0])
    data_d = draw_old_consistency_bland(axs[1, 1])
    fig.suptitle("SI Figure S7. Former main-text consistency figure moved to the SI", fontsize=20, fontweight="bold", y=1.01)
    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Matched-pair consistency diagnostics between ASR and FSR; formerly the main-text Figure 2.", ["datasets.matched_pairs"], ["ASR__Solvent_stability", "FSR__Solvent_stability", "ASR__Water_stability", "FSR__Water_stability", "ASR__Thermal_stability (℃)", "FSR__Thermal_stability (℃)"])
    assets = save_composite_assets(figure_name, "si", fig, {"panel_a": data_a, "panel_b": data_b, "panel_c": data_c, "panel_d": data_d}, fig_meta)
    plt.close(fig)
    panels = {
        "a": render_panel(draw_old_consistency_scatter_solvent, figure_name, "panel_a_solvent_scatter", figsize=(7.2, 5.8)),
        "b": render_panel(draw_old_consistency_scatter_water, figure_name, "panel_b_water_scatter", figsize=(7.2, 5.8)),
        "c": render_panel(draw_old_consistency_threshold, figure_name, "panel_c_threshold_agreement", figsize=(8.0, 6.2)),
        "d": render_panel(draw_old_consistency_bland, figure_name, "panel_d_bland_altman", figsize=(7.2, 5.8)),
    }
    return {"assets": assets, "panels": panels, "meta": fig_meta}


def build_si_s8_importance() -> Dict[str, Any]:
    figure_name = "SI_Figure_S8_importance"
    fig, ax = plt.subplots(figsize=(12.0, 7.2), constrained_layout=True)
    data = draw_importance_panel(ax, panel_letter="(S8)")
    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Permutation importance ranking for the strongest ASR water-stability regime.", ["tables.permutation_importance_df"], ["dataset", "target", "regime", "model", "feature", "importance_mean", "importance_std"])
    assets = save_composite_assets(figure_name, "si", fig, {"importance": data}, fig_meta)
    plt.close(fig)
    panel = render_panel(lambda ax: draw_importance_panel(ax, panel_letter="(S8)"), figure_name, "panel_s8_importance", figsize=(10.0, 6.4))
    return {"assets": assets, "panels": {"main": panel}, "meta": fig_meta}


def build_si_s9_failure_gallery() -> Dict[str, Any]:
    figure_name = "SI_Figure_S9_failure_gallery"
    fig, axs = plt.subplots(2, 2, figsize=(15.0, 11.5), constrained_layout=True)

    # A target counts
    ax = axs[0,0]
    panel_tag(ax, "(a)", x=-0.10, y=1.05)
    ax.set_title("Failure cases by target", loc="left", fontsize=15, pad=10)
    a = failure_cases_df.groupby("target", as_index=False).size().rename(columns={"size": "count"})
    sns.barplot(data=a, x="target", y="count", ax=ax, palette=[PALETTE["blue"], PALETTE["teal"], PALETTE["brick"]])
    ax.set_xticklabels([target_label(t.get_text()) for t in ax.get_xticklabels()], rotation=18)

    # B metal enrichment
    ax = axs[0,1]
    panel_tag(ax, "(b)", x=-0.10, y=1.05)
    ax.set_title("Failure enrichment by metal", loc="left", fontsize=15, pad=10)
    b = failure_cases_df.groupby("primary_metal", as_index=False).size().rename(columns={"size": "count"}).sort_values("count", ascending=False).head(12)
    ranked_barh(ax, b, y="primary_metal", x="count", title="Failure enrichment by metal", color=PALETTE["brick"], xlabel="Count")

    # C absolute error distribution
    ax = axs[1,0]
    panel_tag(ax, "(c)", x=-0.10, y=1.05)
    ax.set_title("Absolute-error distribution among archived failures", loc="left", fontsize=15, pad=10)
    c = failure_cases_df.copy()
    c["abs_error"] = (c["y_true"] - c["y_pred"]).abs()
    sns.histplot(c["abs_error"], bins=16, color=PALETTE["plum"], ax=ax)
    ax.set_xlabel("Absolute error")

    # D topology enrichment
    ax = axs[1,1]
    panel_tag(ax, "(d)", x=-0.10, y=1.05)
    ax.set_title("Failure enrichment by topology", loc="left", fontsize=15, pad=10)
    d = failure_cases_df.groupby("topology(AllNodes)", as_index=False).size().rename(columns={"size": "count"}).sort_values("count", ascending=False).head(12)
    ranked_barh(ax, d, y="topology(AllNodes)", x="count", title="Failure enrichment by topology", color=PALETTE["olive"], xlabel="Count")

    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Failure-case diagnostics across targets, metals, absolute-error spread, and topology enrichment.", ["tables.failure_cases_df"], ["target", "primary_metal", "topology(AllNodes)", "y_true", "y_pred"])
    assets = save_composite_assets(figure_name, "si", fig, {"panel_a": a, "panel_b": b, "panel_c": c, "panel_d": d}, fig_meta)
    plt.close(fig)
    return {"assets": assets, "meta": fig_meta}


def build_si_s10_screening_subset() -> Dict[str, Any]:
    figure_name = "SI_Figure_S10_screening_subset"
    fig, ax = plt.subplots(figsize=(12.5, 7.5), constrained_layout=True)
    data = draw_screening_subset_line_panel(ax, panel_letter="(S10)")
    fig_meta = add_build_note({"figure_name": figure_name, "kind": "si"}, "Cutoff-dependent shortlist utility in the recommended-screening subset for ASR solvent and water stability.", ["tables.shortlist_df"], ["dataset", "target", "screening_cutoff", "subset", "ROC_AUC"])
    assets = save_composite_assets(figure_name, "si", fig, {"screening_subset": data}, fig_meta)
    plt.close(fig)
    panel = render_panel(lambda ax: draw_screening_subset_line_panel(ax, panel_letter="(S10)"), figure_name, "panel_s10_screening_subset", figsize=(11.5, 6.8))
    return {"assets": assets, "panels": {"main": panel}, "meta": fig_meta}

In [ ]:
# ============================================================================
# 7. TABLES

In [10]:
# ============================================================================
def build_tables() -> Dict[str, Any]:
    tables_out = {}

    dataset_overview = pd.DataFrame([
        {"Dataset": "ASR", "Rows": len(asr_clean), "Columns": asr_clean.shape[1], "Role": "Core benchmark"},
        {"Dataset": "FSR", "Rows": len(fsr_clean), "Columns": fsr_clean.shape[1], "Role": "Companion benchmark"},
        {"Dataset": "ION", "Rows": len(ion_clean), "Columns": ion_clean.shape[1], "Role": "Audit-only dataset"},
        {"Dataset": "Matched pairs", "Rows": len(matched_pairs), "Columns": matched_pairs.shape[1], "Role": "ASR–FSR overlap audit"},
    ])
    tables_out["dataset_overview"] = export_table(dataset_overview, "table_dataset_overview")

    best_reg = (
        regression_summary.sort_values(["dataset", "target", "Spearman", "R2", "RMSE"], ascending=[True, True, False, False, True])
        .groupby(["dataset", "target"], as_index=False)
        .head(1)
        .copy()
    )
    best_reg["Target"] = best_reg["target"].map(target_label)
    best_reg["Regime"] = best_reg["regime"].map(regime_label)
    best_reg = best_reg[["dataset", "Target", "Regime", "model", "RMSE", "MAE", "R2", "Spearman"]]
    tables_out["best_regression"] = export_table(best_reg, "table_best_regression")

    best_cls = (
        classification_summary.sort_values(["dataset", "target", "screening_cutoff", "ROC_AUC", "PR_AUC"], ascending=[True, True, True, False, False])
        .groupby(["dataset", "target", "screening_cutoff"], as_index=False)
        .head(1)
        .copy()
    )
    best_cls["Target"] = best_cls["target"].map(target_label)
    best_cls["Regime"] = best_cls["regime"].map(regime_label)
    best_cls = best_cls[["dataset", "Target", "screening_cutoff", "Regime", "model", "ROC_AUC", "PR_AUC", "Balanced_Accuracy", "top_confidence_precision"]]
    tables_out["best_classification"] = export_table(best_cls, "table_best_classification")

    tables_out["regression_summary_full"] = export_table(regression_summary, "table_regression_summary_full")
    tables_out["classification_summary_full"] = export_table(classification_summary, "table_classification_summary_full")
    tables_out["permutation_importance_full"] = export_table(permutation_importance_df, "table_permutation_importance_full")
    tables_out["shortlist_full"] = export_table(shortlist_df, "table_shortlist_full")
    tables_out["audit_full"] = export_table(audit_df, "table_audit_full")

    return tables_out

In [ ]:
# ============================================================================
# 8. MAIN ORCHESTRATION

In [11]:
# ============================================================================
def main() -> None:
    build_manifest: Dict[str, Any] = {
        "config": {
            "PICKLE_PATH": PICKLE_PATH,
            "OUT_DIR": str(OUT_DIR),
            "DPI": DPI,
            "SAVE_PNG": SAVE_PNG,
            "SAVE_PDF": SAVE_PDF,
        },
        "archive_metadata": metadata,
        "directories": {k: str(v) for k, v in DIRS.items()},
        "main_figures": {},
        "si_figures": {},
        "tables": {},
        "notes": {
            "moved_old_main_figure_2_to_si": True,
            "new_main_figure_4_from_S3_S5_S8_S10": True,
            "downstream_renumbering": {
                "new_figure_4": "descriptor interdependence, threshold robustness, feature importance, and shortlist applicability",
                "new_figure_5": "former chemistry-facing Figure 4",
                "new_figure_6": "former application-facing Figure 5",
            },
        },
    }

    save_json(build_manifest["config"], DIRS["metadata"] / "run_config.json")
    save_json(metadata, DIRS["metadata"] / "archive_metadata.json")

    # Main-text figures
    build_manifest["main_figures"]["Figure_1_dataset_architecture"] = build_figure_1_dataset_architecture()
    build_manifest["main_figures"]["Figure_3_descriptor_sufficiency"] = build_figure_3_descriptor_sufficiency()
    build_manifest["main_figures"]["Figure_4_descriptor_interdependence_robustness_utility"] = build_figure_4_descriptor_structure_robustness_utility()
    build_manifest["main_figures"]["Figure_5_chemistry_regime_map"] = build_figure_5_chemistry_facing()
    build_manifest["main_figures"]["Figure_6_practical_screening"] = build_figure_6_application_facing()

    # SI figures
    build_manifest["si_figures"]["S1_missingness"] = build_si_s1_missingness()
    build_manifest["si_figures"]["S2_descriptor_distributions"] = build_si_s2_descriptor_distributions()
    build_manifest["si_figures"]["S3_correlation_matrix"] = build_si_s3_correlation_matrix()
    build_manifest["si_figures"]["S4_metal_breakdown"] = build_si_s4_metal_breakdown()
    build_manifest["si_figures"]["S5_threshold_robustness"] = build_si_s5_threshold_robustness()
    build_manifest["si_figures"]["S6_grouped_cv_details"] = build_si_s6_grouped_cv_details()
    build_manifest["si_figures"]["S7_matched_pair_diagnostics"] = build_si_s7_old_main_figure2_consistency()
    build_manifest["si_figures"]["S8_importance"] = build_si_s8_importance()
    build_manifest["si_figures"]["S9_failure_gallery"] = build_si_s9_failure_gallery()
    build_manifest["si_figures"]["S10_screening_subset"] = build_si_s10_screening_subset()

    build_manifest["tables"] = build_tables()

    save_json(build_manifest, DIRS["manifests"] / "project15_revised_build_manifest.json")

    print("=" * 100)
    print("PROJECT 15 REVISED FIGURE/TABLE BUILD COMPLETE")
    print("=" * 100)
    print(f"Outputs written to: {OUT_DIR}")
    for name, path in DIRS.items():
        print(f"- {name:16s}: {path}")
    print(f"- manifest         : {DIRS['manifests'] / 'project15_revised_build_manifest.json'}")


if __name__ == "__main__":
    main()

/tmp/ipykernel_17615/1058718334.py:82: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x="Group", y="Count", ax=ax, palette=[DATASET_COLORS["ASR"], DATASET_COLORS["FSR"], PALETTE["teal"]])
/tmp/ipykernel_17615/1058718334.py:82: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x="Group", y="Count", ax=ax, palette=[DATASET_COLORS["ASR"], DATASET_COLORS["FSR"], PALETTE["teal"]])
/tmp/ipykernel_17615/1058718334.py:179: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([target_label(t.get_text()) for t in ax.get_xticklabels()], rotation=18)
/tmp/ipykernel_17615/1058718334.py:179: User

PROJECT 15 REVISED FIGURE/TABLE BUILD COMPLETE
Outputs written to: /content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_extracted
- manifests       : /content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_extracted/00_manifests
- metadata        : /content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_extracted/01_metadata
- panel_data      : /content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_extracted/02_panel_data
- panel_metadata  : /content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_extracted/03_panel_metadata
- panel_png       : /content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_extracted/04_panel_png
- panel_pdf       : /content/drive/MyDrive/Papers_DT/Paper15_AAA/V1/Codes/Ver1/Redrawing_Figure_V2/project15_extracted/05_panel_pdf
- figure_data     : /content/drive/My

## Final note

After running the notebook, inspect:

- `00_manifests/`
- `03_panel_metadata/`
- `04_panel_png/`
- `06_figure_data/`
- `08_main_figures_png/`
- `10_si_figures_png/`

Those folders contain the panel-level and figure-level assets intended for later manual editing.
